# In Silico Saturation Mutagenesis → HDF5
For each position in a region, substitute every non-reference base and record the change in model predictions.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import torch
import h5py
from pathlib import Path

from src.infer._utils import load_model, parse_bed, parse_region, fetch_one_hot
from src.data.data_utils import GenomeSequence

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT     = 'outputs/basenji2_wt/checkpoints/best_model.pt'
CONFIG         = 'src/configs/transformer_wt.yaml'
FASTA          = '/path/to/genome.fa'

# Provide either REGION (single) or BED (multiple)
REGION         = 'chr01:1000000-1001000'   # set to None to use BED
BED            = None                       # e.g. 'data/regions.bed'

MUT_BATCH_SIZE = 64
OUTPUT         = 'output/ism_scores.h5'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']
genome = GenomeSequence(FASTA)
print(f'Model loaded on {device}')

if BED:
    regions = list(parse_bed(BED))
else:
    chrom, start, end = parse_region(REGION)
    regions = [(chrom, start, end, REGION, '+')]
print(f'{len(regions)} region(s)')

In [ ]:
def ism_region(model, one_hot_np, device, mut_batch_size):
    L = one_hot_np.shape[1]
    ref_t = torch.tensor(one_hot_np[None], dtype=torch.float32).to(device)
    with torch.no_grad():
        ref_pred = model(ref_t)['phase_pred'].squeeze(0).cpu().numpy()  # [4]

    ism_scores = np.zeros((L, 4, 4), dtype=np.float32)
    mut_list = []
    for i in range(L):
        ref_base = one_hot_np[:, i].argmax() if one_hot_np[:, i].sum() > 0 else -1
        for b in range(4):
            if b == ref_base:
                continue
            mut = one_hot_np.copy()
            mut[:, i] = 0.0
            mut[b, i] = 1.0
            mut_list.append((i, b, mut))

    for bs in range(0, len(mut_list), mut_batch_size):
        batch = mut_list[bs: bs + mut_batch_size]
        x = torch.tensor(np.stack([m[2] for m in batch]), dtype=torch.float32).to(device)
        with torch.no_grad():
            preds = model(x)['phase_pred'].cpu().numpy()
        for k, (i, b, _) in enumerate(batch):
            ism_scores[i, b, :] = preds[k] - ref_pred

    ism_scores -= ism_scores.mean(axis=1, keepdims=True)
    scd = np.sqrt((ism_scores ** 2).sum(axis=1))  # [L, 4]
    return ref_pred, ism_scores, scd

In [ ]:
n, L = len(regions), window_size
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)

with h5py.File(OUTPUT, 'w') as hf:
    hf.create_dataset('seqnames', data=np.array([r[3] for r in regions], dtype='S'))
    hf.create_dataset('chrom',    data=np.array([r[0] for r in regions], dtype='S'))
    hf.create_dataset('start',    data=np.array([r[1] for r in regions], dtype=np.int32))
    hf.create_dataset('end',      data=np.array([r[2] for r in regions], dtype=np.int32))
    hf.create_dataset('strand',   data=np.array([r[4] for r in regions], dtype='S'))
    ds_seqs = hf.create_dataset('seqs',       shape=(n, 4, L),    dtype=np.float32)
    ds_ref  = hf.create_dataset('ref_pred',   shape=(n, 4),       dtype=np.float32)
    ds_ism  = hf.create_dataset('ism_scores', shape=(n, L, 4, 4), dtype=np.float32)
    ds_scd  = hf.create_dataset('scd',        shape=(n, L, 4),    dtype=np.float32)

    for idx, (chrom, start, end, name, strand) in enumerate(regions):
        print(f'[{idx+1}/{n}] {name}')
        oh = fetch_one_hot(genome, chrom, start, end, window_size, strand)
        ref_pred, ism_scores, scd = ism_region(model, oh, device, MUT_BATCH_SIZE)
        ds_seqs[idx] = oh
        ds_ref[idx]  = ref_pred
        ds_ism[idx]  = ism_scores
        ds_scd[idx]  = scd

print(f'Written ISM scores to {OUTPUT}')

## Quick visualization
Plot SCD (sequence contribution score) for the first region.

In [ ]:
import matplotlib.pyplot as plt

with h5py.File(OUTPUT, 'r') as hf:
    scd = hf['scd'][0]   # [L, 4] — phases: ES, MS, LS, G1

fig, axes = plt.subplots(4, 1, figsize=(16, 8), sharex=True)
for i, (ax, name) in enumerate(zip(axes, ['ES', 'MS', 'LS', 'G1'])):
    ax.fill_between(range(len(scd)), scd[:, i], alpha=0.7)
    ax.set_ylabel(name)
axes[-1].set_xlabel('Position')
fig.suptitle('ISM SCD scores')
plt.tight_layout()
plt.show()